In [4]:
import pandas as pd
import numpy as np
import os
import re

# конфигурация
DATA_PATH = 'data/'

# Предкомпилированный паттерн для поиска суммы. 
# Ищет цифры (возможно с точкой/запятой) и варианты обозначения рублей.
AMOUNT_PATTERN = re.compile(r'(\d+(?:[.,]\d+)?)\s*(?:р\.|руб\.|руб|рублей|₽|р)', flags=re.IGNORECASE)

class DataLoader:
    """Класс для загрузки файлов и первичной очистки данных (исправление опечаток, форматов)."""
    
    def __init__(self, data_path: str):
        self.data_path = data_path
        self.data = {}
        
    def _clean_phone(self, phone_series: pd.Series) -> pd.Series:
        """Внутренний метод: убирает все знаки (плюсы, скобки) из номера телефона."""
        return phone_series.astype(str).str.replace(r'\D', '', regex=True)

    def load_and_clean(self):
        
        # 1. Клиенты банка (Bank Clients)
        bc = pd.read_csv(os.path.join(self.data_path, 'bank_clients.tsv'), sep='\t')
        bc.rename(columns={'accout': 'account'}, inplace=True) # ИСПРАВЛЕНА ОПЕЧАТКА accout -> account
        bc['phone'] = self._clean_phone(bc['phone'])
        self.data['bank_clients'] = bc
        
        # 2. Жалобы (Bank Complaints)
        comp = pd.read_csv(os.path.join(self.data_path, 'bank_complaints.tsv'), sep='\t')
        comp.rename(columns={'uerId': 'userId'}, inplace=True) # ИСПРАВЛЕНА ОПЕЧАТКА uerId -> userId
        comp['event_date'] = pd.to_datetime(comp['event_date'])
        self.data['bank_complaints'] = comp
        
        # 3. Транзакции (Bank Transactions)
        tx = pd.read_csv(os.path.join(self.data_path, 'bank_transactions.tsv'), sep='\t')
        tx['event_date'] = pd.to_datetime(tx['event_date'])
        self.data['bank_transactions'] = tx
        
        # 4. Мобильные клиенты (Mobile Clients)
        mc = pd.read_csv(os.path.join(self.data_path, 'mobile_clients.tsv'), sep='\t')
        mc['phone'] = self._clean_phone(mc['phone'])
        self.data['mobile_clients'] = mc
        
        # 5. Детализация звонков (Mobile Build)
        mb = pd.read_csv(os.path.join(self.data_path, 'mobile_build.tsv'), sep='\t')
        mb['event_date'] = pd.to_datetime(mb['event_date'])
        mb['from_call'] = self._clean_phone(mb['from_call'])
        mb['to_call'] = self._clean_phone(mb['to_call'])
        self.data['mobile_build'] = mb
        
        # 6. Экосистема (Ecosystem Mapping)
        eco = pd.read_csv(os.path.join(self.data_path, 'ecosystem_mapping.tsv'), sep='\t')
        eco.rename(columns={'market_plece_user_id': 'market_place_user_id'}, inplace=True) # ОПЕЧАТКА plece -> place
        # Заменяем текстовые '\N' на настоящий NaN
        eco.replace(r'\\N', np.nan, regex=True, inplace=True) 
        self.data['ecosystem'] = eco
        
        # 7. Маркетплейс Доставки (Market Place Delivery)
        md = pd.read_csv(os.path.join(self.data_path, 'market_place_delivery.tsv'), sep='\t')
        md['event_date'] = pd.to_datetime(md['event_date'])
        md['contact_phone'] = self._clean_phone(md['contact_phone'])
        self.data['market_delivery'] = md
        
        return self.data


class FraudDetector:
    """Класс для выявления полного пути мошенника через pandas merge (векторизация)."""
    
    def __init__(self, data_dict: dict):
        self.data = data_dict
        self.market_details = {} # Для сохранения детализации доставок
        self.calls_details = pd.DataFrame() # Для сохранения таблицы звонков
        
    def _extract_amount(self, text: str):
        """Парсинг суммы из текста жалобы."""
        if pd.isna(text): return np.nan
        match = AMOUNT_PATTERN.search(str(text))
        if match:
            return float(match.group(1).replace(',', '.'))
        return np.nan

    def run_investigation(self) -> pd.DataFrame:
        """Основной алгоритм расследования мошенничества по шагам."""
        print("🔍 Запуск алгоритма расследования...")
        
        # ШАГ 1: Извлекаем сумму кражи
        df = self.data['bank_complaints'].copy()
        df['extracted_amount'] = df['text'].apply(self._extract_amount)
        # ИСПРАВЛЕНИЕ: переименовываем text в complaint_text
        df = df.dropna(subset=['extracted_amount']).rename(columns={
            'event_date': 'complaint_date',
            'text': 'complaint_text' 
        })
        
        # ШАГ 2: Подтягиваем счет и телефон жертвы
        df = pd.merge(df, self.data['bank_clients'], on='userId', how='inner')
        # ИСПРАВЛЕНИЕ: сразу переименовываем fio в victim_fio, чтобы не было конфликта дальше
        df.rename(columns={
            'account': 'victim_account', 
            'phone': 'victim_phone',
            'fio': 'victim_fio'
        }, inplace=True)
        
        # ШАГ 3: Ищем транзакцию (по счету жертвы и сумме)
        tx = self.data['bank_transactions']
        df = pd.merge(df, tx, left_on=['victim_account', 'extracted_amount'], right_on=['account_out', 'value'], how='inner')
        df.rename(columns={'event_date': 'transaction_date', 'account_in': 'fraud_account'}, inplace=True)
        
        # Логическое условие: транзакция должна быть ДО жалобы
        df = df[df['transaction_date'] <= df['complaint_date']]
        df = df.sort_values('transaction_date', ascending=False).drop_duplicates(subset=['userId', 'complaint_date'])
        
        # ШАГ 4: Ищем звонок от мошенника (кто звонил жертве ДО транзакции)
        calls = self.data['mobile_build']
        df = pd.merge(df, calls, left_on='victim_phone', right_on='to_call', how='inner')
        df.rename(columns={'event_date': 'call_date', 'from_call': 'fraud_phone'}, inplace=True)
        
        # Логическое условие: звонок был ДО перевода денег
        df = df[df['call_date'] <= df['transaction_date']]
        df = df.sort_values('call_date', ascending=False).drop_duplicates(subset=['userId', 'complaint_date'])
        
        self.calls_details = df[['call_date', 'fraud_phone', 'victim_phone', 'duration_sec']].copy()
        
        # ШАГ 5: Узнаем профиль мошенника в мобильной сети
        mc = self.data['mobile_clients']
        df = pd.merge(df, mc[['phone', 'client_id', 'fio']], left_on='fraud_phone', right_on='phone', how='left')
        # Теперь конфликта нет, fio корректно переименуется в fraud_fio
        df.rename(columns={'client_id': 'fraud_mobile_id', 'fio': 'fraud_fio'}, inplace=True)
        
        # ШАГ 6: Связываем с Экосистемой (Маркетплейс)
        eco = self.data['ecosystem']
        df = pd.merge(df, eco[['mobile_user_id', 'market_place_user_id']], left_on='fraud_mobile_id', right_on='mobile_user_id', how='left')
        
        # ШАГ 7: Проверяем заказы на Маркетплейсе
        md = self.data['market_delivery']
        market_ids = df['market_place_user_id'].dropna().unique()
        
        if len(market_ids) > 0:
            market_activity = md[md['user_id'].isin(market_ids)]
            self.market_details = dict(tuple(market_activity.groupby('user_id')))
            counts = market_activity.groupby('user_id').size().to_dict()
            df['market_deliveries'] = df['market_place_user_id'].map(counts).fillna(0)
        else:
            df['market_deliveries'] = 0
            
        df['has_market_activity'] = (df['market_deliveries'] > 0).astype(int)
        df['has_calls'] = 1
        
        final_cols =[
            'userId', 'complaint_text', 'complaint_date', 'extracted_amount', 
            'victim_account', 'victim_phone', 'transaction_date', 'fraud_account', 
            'fraud_phone', 'fraud_fio', 'market_place_user_id', 'has_calls', 'market_deliveries'
        ]
        
        return df[final_cols]

class DataExporter:
    """Класс для генерации файлов для Neo4j и сохранения итоговых CSV."""
    
    def __init__(self, data_path: str):
        self.data_path = data_path
        # Убедимся, что папка для выгрузки существует
        os.makedirs(self.data_path, exist_ok=True)
        
    def export_neo4j(self, fraud_df: pd.DataFrame):
        print("🌐 Подготовка графов для Neo4j...")
        nodes =[]
        edges =[]
        
        for row in fraud_df.itertuples():
            # Узел 1: Жертва
            nodes.append({
                'id': f"victim_{row.userId}", 'type': 'person', 'role': 'victim',
                'bank_id': row.userId, 'account': row.victim_account, 'phone': row.victim_phone
            })
            
            # Узел 2: Мошенник (если идентифицирован телефон)
            if pd.notna(row.fraud_phone):
                nodes.append({
                    'id': f"fraud_phone_{row.fraud_phone}", 'type': 'person', 'role': 'fraudster',
                    'bank_id': None, 'account': row.fraud_account, 
                    'phone': row.fraud_phone, 'fio': row.fraud_fio
                })
                
                # Связь: Перевод денег
                edges.append({
                    'from': f"victim_{row.userId}", 'to': f"fraud_phone_{row.fraud_phone}",
                    'type': 'TRANSFERRED', 'amount': row.extracted_amount, 'date': row.transaction_date
                })
                
                # Связь: Звонок
                edges.append({
                    'from': f"fraud_phone_{row.fraud_phone}", 'to': f"victim_{row.userId}",
                    'type': 'CALLED', 'amount': None, 'date': None # Дату звонка можно подтянуть при желании
                })
                
        # Дедубликация узлов
        nodes_df = pd.DataFrame(nodes).drop_duplicates(subset=['id'])
        edges_df = pd.DataFrame(edges)
        
        nodes_df.to_csv(os.path.join(self.data_path, 'neo4j_nodes.csv'), index=False)
        edges_df.to_csv(os.path.join(self.data_path, 'neo4j_edges.csv'), index=False)

    def export_reports(self, fraud_df: pd.DataFrame, detector: FraudDetector):
        # Сохраняем главный файл
        main_path = os.path.join(self.data_path, 'all_fraud_cases.csv')
        fraud_df.to_csv(main_path, index=False)
        
        # Сохраняем звонки
        if not detector.calls_details.empty:
            calls_path = os.path.join(self.data_path, 'fraud_calls.csv')
            detector.calls_details.to_csv(calls_path, index=False)
            
        # Сохраняем детализацию доставок каждого мошенника
        for mp_id, df_deliv in detector.market_details.items():
            deliv_path = os.path.join(self.data_path, f'fraud_delivery_{mp_id}.csv')
            df_deliv.to_csv(deliv_path, index=False)
            
if __name__ == "__main__":
    # 1. Загрузка
    loader = DataLoader(DATA_PATH)
    raw_data = loader.load_and_clean()
    
    # 2. Детектирование
    detector = FraudDetector(raw_data)
    fraud_cases = detector.run_investigation()
    
    # 3. Печать результатов в консоль/ноутбук
    print("\n" + "="*50)
    print(f"📄 Всего жалоб в базе: {len(raw_data['bank_complaints'])}")
    print(f"🚨 Найдено цепочек мошенничества: {len(fraud_cases)}")
    print("="*50 + "\n")
    
    if len(fraud_cases) > 0:
        print("КРАТКОЕ ДОСЬЕ НА МОШЕННИКОВ:")
        # Выведем красиво в табличном виде
        display(fraud_cases[['userId', 'extracted_amount', 'fraud_account', 'fraud_fio', 'fraud_phone', 'market_deliveries']])
        
        print("\nАДРЕСА ДОСТАВОК МОШЕННИКОВ (из Маркетплейса):")
        for mp_id, deliveries in detector.market_details.items():
            print(f"\nМаркетплейс ID: {mp_id}")
            display(deliveries[['event_date', 'contact_fio', 'contact_phone', 'address']].head(3))
            
    # 4. Экспорт
    print("\n")
    exporter = DataExporter(DATA_PATH)
    exporter.export_neo4j(fraud_cases)
    exporter.export_reports(fraud_cases, detector)

🔍 Запуск алгоритма расследования...

📄 Всего жалоб в базе: 1
🚨 Найдено цепочек мошенничества: 1

КРАТКОЕ ДОСЬЕ НА МОШЕННИКОВ:


,userId,extracted_amount,fraud_account,fraud_fio,fraud_phone,market_deliveries
0,b2857395394,15000.0,40101810045250011152,Иванов В.Г.,79297654321,3



АДРЕСА ДОСТАВОК МОШЕННИКОВ (из Маркетплейса):

Маркетплейс ID: mp2920493810


,event_date,contact_fio,contact_phone,address
0,2020-05-07 15:21:45,Иванов В.Г.,79297483321,"г. Москва, ул. Маршала Жукова, д. 7, офис 12"
1,2020-05-12 16:01:23,Иванов В.Г.,79297483321,"г. Москва, ул. Маршала Жукова, д. 7, офис 12"
2,2020-06-01 22:39:18,Иванова Д.Е.,79297483321,"г. Москва, Проспект мира, д. 12 кв. 175"




🌐 Подготовка графов для Neo4j...
